## Corpus

<div style="font-size: 0.85em">

**The main email data** (all loaded together from `main_email_dfs_20251007.Rdata`; this notebook reads `msgs10` and `recip4` as CSV files from `data/`)

| Object | Rows × cols | What it is |
|---|---|---|
| `msgs10` | 34,978 × 32 | **Messages, latest version.** One row per email. Drops the link to the original message ID (`msgid1`), so it has more duplicates, flagged in `del_new`. Includes manual cleaning from `msgr_grp2 (1).xlsx`. This is what the EDA script analyzes. |
| `msgs8` | 39,579 × 31 | **Messages, earlier version.** Keeps `msgid1`, useful for tracing a message back or auditing. |
| `recip4` | 129,654 × 4 | **Recipients, latest version.** One row per recipient per email (to/cc). Drops `msgid1`, so fewer duplicates. Pairs with `msgs10`. |
| `recip3` | 162,511 × 8 | **Recipients, earlier version.** Keeps `msgid1` and extra columns for troubleshooting. Pairs with `msgs8`. |
| `crs` | 403 × 9 | **CITF researchers.** Links individual investigators and staff to their CITF study IDs. People can appear more than once if they're on several studies. |

</div>

##### 1 - Load email data

Loads `msgs10` (messages) and `recip4` (recipients) from `data/` and checks missingness in each.

In [ ]:
import pandas as pd
from pathlib import Path

# Repository root (the folder with _quarto.yml), found from this notebook's folder
root = Path.cwd()
while not (root / '_quarto.yml').exists():
    root = root.parent
here = root / 'corpus'

msgs  = pd.read_csv(root / 'data' / 'msgs10.csv')   # 34,978 emails
recip = pd.read_csv(root / 'data' / 'recip4.csv')   # 129,654 recipient rows
counts = {'Start (msgs10)': len(msgs)}

def missingness(df, name):
    out = pd.DataFrame({
        'dtype': df.dtypes,
        'n_missing': df.isna().sum(),
        'pct_missing': (df.isna().mean() * 100).round(1)
    })
    print(f"\n{name}: {df.shape[0]:,} rows × {df.shape[1]} cols")
    display(out)

missingness(msgs, 'msgs10')
missingness(recip, 'recip4')

##### 2 - Flatten emails

Builds a one-row-per-email table (from, to, cc, subject, body) by joining the messages in `msgs10` with their recipients from `recip4`.

In [ ]:
# Drop flagged duplicates (del_new == 1): -2,258 → 32,720 emails
msgs = msgs[msgs['del_new'] != 1]
counts['After del_new filter'] = len(msgs)

# Keep only recipient rows that have an email address: -2,724 → 126,930 rows
recip = recip.dropna(subset=['recipient_email_clean'])

# One row per email: join all 'to' and all 'cc' email addresses into a single string
to = recip[recip['recipient_type'] == 'to'].groupby('msgid2')['recipient_email_clean'].agg('; '.join).rename('to')
cc = recip[recip['recipient_type'] == 'cc'].groupby('msgid2')['recipient_email_clean'].agg('; '.join).rename('cc')

# Left joins, no rows lost: 32,720 emails
emails = (msgs[['msgid2', 'msg_citfid', 'sender_email', 'msg_subject_clean', 'msg_body']]
          .rename(columns={'sender_email': 'from',
                           'msg_subject_clean': 'subject',
                           'msg_body': 'body'})
          .merge(to, on='msgid2', how='left')
          .merge(cc, on='msgid2', how='left')
          [['msgid2', 'msg_citfid', 'from', 'to', 'cc', 'subject', 'body']])
counts['After merges'] = len(emails)

# Drop emails with identical content under different IDs: -3,240 → 29,480 emails
content_cols = ['from', 'to', 'cc', 'subject', 'body']
emails = emails.drop_duplicates(subset=content_cols, keep='first')
counts['After content dedup'] = len(emails)

# Keep only emails with a sender, at least one recipient, and a body: -3,260 → 26,220 emails
emails = emails.dropna(subset=['from', 'to', 'body'])
counts['After from/to/body filter'] = len(emails)

# Summary
summary = pd.DataFrame({'emails': counts})
summary['removed'] = summary['emails'].shift() - summary['emails']
print(summary)

# Columns: N non-missing (% missing)
for col in emails.columns:
    n = emails[col].notna().sum()
    pct = emails[col].isna().mean() * 100
    print(f"{col:<10} {n:>6,} ({pct:.1f}% missing)")

display(emails.head())

##### 3 - Restrict to emails linked to a study

Keeps only emails with a valid CITF study ID in `msg_citfid` (such as `103-CC`), excluding `200-NF`. Marks the pool, `in_pool`: emails of 300 to 2,000 characters with no credential-like text. The DocuScope assessment and the zones use the pool. Saves the study emails to `emails_study.parquet`.

In [ ]:
# Keep only emails with a valid study ID (e.g. 103-CC), excluding 200-NF: -15,823 → 10,397 emails
valid = (emails['msg_citfid'].astype(str).str.match(r'^\d{3}-[A-Za-z]{2}$')
         & (emails['msg_citfid'].astype(str).str.upper() != '200-NF'))
emails_study = emails[valid]
counts['After study ID filter'] = len(emails_study)

# Summary
summary = pd.DataFrame({'emails': counts})
summary['removed'] = summary['emails'].shift() - summary['emails']
print(summary)

print("\nDistinct study IDs:", emails_study['msg_citfid'].nunique())

# Columns: N non-missing (% missing)
for col in emails_study.columns:
    n = emails_study[col].notna().sum()
    pct = emails_study[col].isna().mean() * 100
    print(f"{col:<10} {n:>6,} ({pct:.1f}% missing)")

display(emails_study.head())

# ---- Pool: emails of a readable length with no credential-like text ----
# Exclude emails that look like they contain credentials
cred_pattern = (r'password|passwd|\bpwd\b|\bpass\s*:|mot de passe|\bmdp\b|username|\bUN\s*:'
                r'|token=|privateurl|access_token|\bsig=|api[_ ]?key')
has_creds = emails_study['body'].str.contains(cred_pattern, case=False, regex=True, na=False)
print("\nEmails excluded for possible credentials:", has_creds.sum())

# Keep emails of a readable length (characters)
min_chars, max_chars = 300, 2000
body_len = emails_study['body'].str.len()
in_range = body_len.between(min_chars, max_chars)
print(body_len.describe())
print(f"Emails between {min_chars} and {max_chars} chars (no credentials):",
      (in_range & ~has_creds).sum())

emails_study = emails_study.assign(in_pool=in_range & ~has_creds)
study_path = here / 'emails_study.parquet'
emails_study.to_parquet(study_path, index=False)
print("Saved to:", study_path)

##### 4 - Label email lines with Chipmunk

Labels every line of every pool email with one of the 15 line labels of Chipmunk (Bevendorff et al. 2020), the email segmenter released with its trained model. DocuScope tags each email's full text as before; the labels only mark which lines are salutation, paragraph, closing, signature, banner (`mua_signature`), quoted text and so on, so that later blocks can place each tag. Chipmunk runs in its own environment (`~/models/email-zoning/chipmunk-env`, TensorFlow 2.2) at the repository commit and package versions recorded beside it, and its output is saved to `email_zones.parquet`. Two labels are kept per line: `chipmunk_raw`, the model's top label, and `chipmunk`, the label after the repository's own correction rules, which the authors applied to build their corpus. Block 5 maps these labels to zones.

In [ ]:
import json, os, subprocess, tempfile, time
import pandas as pd

# Pool from block 3: study emails of 300 to 2,000 characters, no credentials
pool_z = emails_study[emails_study['in_pool']][['msgid2', 'body']].copy()
pool_z['msgid2'] = pool_z['msgid2'].astype(str)
bodies = dict(zip(pool_z['msgid2'], pool_z['body'].astype(str)))

zoning_dir = os.path.expanduser('~/models/email-zoning')
zones_path = str(here / 'email_zones.parquet')

# Chipmunk prediction, as run on the five emails, reading and writing one email per JSON line
chipmunk_script = r'''
import json, sys
import numpy as np
from tensorflow.keras import models
from util.mail_classification import load_fasttext_model, MailLinesSequence
from parsing.message_segmenter import CONTEXT_SHAPE, INF_BATCH_SIZE, _post_process_labels

load_fasttext_model("weights/fasttext-model.bin")
model = models.load_model("weights/segmenter_hinge.epoch-11.val_loss-0.093.h5")
with open(sys.argv[1]) as fin, open(sys.argv[2], "w") as fout:
    for row in fin:
        email = json.loads(row)
        text = email["body"].rstrip("\n")
        try:
            seq = MailLinesSequence(text, CONTEXT_SHAPE, labeled=False, input_is_raw_text=True, batch_size=INF_BATCH_SIZE)
            soft = model.predict(seq)
            raw = [MailLinesSequence.LABEL_MAP_INVERSE[int(np.argmax(s))] for s in soft]
            post = [label for _, label in _post_process_labels(seq, soft.copy())]
            fout.write(json.dumps({"msgid2": email["msgid2"], "raw": raw, "zone": post}) + "\n")
        except Exception as e:
            fout.write(json.dumps({"msgid2": email["msgid2"], "error": repr(e)}) + "\n")
'''

# Email text stays on the CITF storage: the temporary folder sits beside this notebook
start_time = time.time()
with tempfile.TemporaryDirectory(dir=here) as tmp:
    in_path, out_path_z, script_path = (os.path.join(tmp, f) for f in ('in.jsonl', 'out.jsonl', 'chipmunk_predict.py'))
    with open(in_path, 'w', encoding='utf-8') as f:
        for m, b in bodies.items():
            f.write(json.dumps({'msgid2': m, 'body': b}, ensure_ascii=False) + '\n')
    with open(script_path, 'w') as f:
        f.write(chipmunk_script)
    env = dict(os.environ, CUDA_VISIBLE_DEVICES='', TF_CPP_MIN_LOG_LEVEL='3', PYTHONWARNINGS='ignore',
               PYTHONPATH=os.path.join(zoning_dir, 'acl20-crawling-mailing-lists/src'))
    run = subprocess.run([os.path.join(zoning_dir, 'chipmunk-env/bin/python'), script_path, in_path, out_path_z],
                         cwd=zoning_dir, env=env, capture_output=True, text=True, timeout=4 * 3600)
    if run.returncode != 0:
        print(run.stderr[-3000:])
        raise RuntimeError('Chipmunk failed')
    results = [json.loads(l) for l in open(out_path_z, encoding='utf-8')]

# One row per line, with character offsets into the body DocuScope tagged
rows, errors, mismatches = [], [], []
for rec in results:
    if 'error' in rec:
        errors.append(rec)
        continue
    lines = bodies[rec['msgid2']].rstrip('\n').split('\n')
    if len(lines) != len(rec['zone']):
        mismatches.append(rec['msgid2'])
        continue
    pos = 0
    for i, (line, raw, zone) in enumerate(zip(lines, rec['raw'], rec['zone']), 1):
        rows.append({'msgid2': rec['msgid2'], 'line': i, 'start': pos, 'end': pos + len(line),
                     'text': line, 'chipmunk_raw': raw, 'chipmunk': zone})
        pos += len(line) + 1

zones = pd.DataFrame(rows)
zones.to_parquet(zones_path, index=False)

print(f"Chipmunk run time: {(time.time() - start_time) / 60:.1f} minutes")
print("Pool emails:", len(bodies))
print("Emails zoned:", zones['msgid2'].nunique())
print("Emails with errors:", len(errors))
print("Emails where line counts differ:", len(mismatches))
print("Lines:", len(zones))
print()
print(zones['chipmunk'].value_counts().to_string())
print()
print("Saved to:", zones_path)

##### 5 - Refine email zones

Maps Chipmunk's labels to six zones (authored, signature, banner, quoted, markup, other) and compares them line by line with zones assigned by regular expressions. Then refines the zones with five rules:

1. Closing lines that Chipmunk labels signature, banner or other are authored.
2. Authored, other or signature lines with disclaimer wording are banner.
3. Authored lines that do not end a sentence, between signature lines, are signature.
4. Banner or other lines that read as prose, between authored lines, are authored.
5. Short lines without sentence punctuation at the end of the message are signature when a signature line follows them or they hold a name, title, organization or contact detail. A one-word sign-off name stays authored, and so does a two-word name under a closing with no credentials and nothing of the signature after it.

Zones from regular expressions then take precedence, in this order: markup, attachment and meeting-invitation lines (other), banner, and quoted, which runs from the first reply or forward header to the end of the email. Saves each line's zone and the rule that set it to `email_zones.parquet`.

In [ ]:
import re
from collections import Counter
import pandas as pd

def show(df):
    print(df.to_string(max_colwidth=90))

zones = pd.read_parquet(zones_path)

# ---- Pool: block 3 emails, minus automated senders ----
automated_senders = [r"docusign", r"no-?reply", r"mailer-daemon", r"postmaster", r"microsoft"]
pool10 = emails_study[emails_study['in_pool']][['msgid2', 'from', 'body']].copy()
pool10['msgid2'] = pool10['msgid2'].astype(str)
pool10 = pool10[~pool10['from'].astype(str).str.contains('|'.join(automated_senders), case=False, regex=True)]

# ---- Zone patterns, with the header and banner forms the comparison found missing ----
quote_patterns = [
    r"\b(on|le)\s[^\n]{4,200}?(\n[^\n]{0,200}?)?\s(wrote|a écrit)\s*:",
    r"\bwrote on [^\n]{4,60}:",
    r"^\s*-{2,}\s*(original message|original appointment|message d['’]origine|forwarded message|message transféré)",
    r"^\s*begin forwarded message\s*:",
    r"^\s*\*?(from|de)\*?\s*:[^\n]*\n(?:[^\n]*\n){0,3}?\s*\*?(sent|date|envoyé)\*?\s*:",
    r"^\s*_{10,}\s*\n(?:[^\n]*\n){0,3}?\s*\*?(from|de)\*?\s*:",
]
banner_patterns = [
    r"caution\s*!?\s*this (message|email) was sent from outside",
    r"\[?caution\s*:\s*(non-[\w-]+ email|external (mail|email))",
    r"caution\s*:\s*external sender",
    r"some people who received this message don['’]?t often get email",
    r"you don['’]?t often get email from",
    r"learn why this is important",
    r"(avertissement automatis|courriel externe|faites preuve de prudence|ne cliquez (pas|sur))",
    r"exercise caution when opening",
    r"do not click (on )?links",
    r"this (message|email) (originated|came) from outside",
    r"work from home alert",
    r"(located on|acknowledge|homeland)[^\n]{0,200}(traditional|ancestral|unceded|treaty|original)[^\n]{0,80}(territor|lands?\b)",
    r"exp[ée]diteur externe",
    r"certaines personnes qui ont re[çc]u ce (courrier|message)",
    r"^\s*notice\s*:\s*the information (transmitted|contained)",
    r"^\s*confidential(ity)? (warning|notice)\s*:?\s*$",
    r"confidentialit[ée]\s*/\s*confidentiality",
    r"^\W*working remotely during covid",
]
closing = (r"(best|best regards|kind regards|warm regards|regards|cheers|all the best|best wishes|"
           r"sincerely|merci|cordialement|(many )?(thanks|thank you)( so much| again)?"
           r"( for your (time|help|patience))?)")
signoff_pattern = re.compile(
    rf"^\s*((thanks|thank you)( again)?,? and )?{closing}\s*[,.!]*\s*([A-Z][\w'’.-]*(\s[A-Z][\w'’.-]*){{0,2}})?\s*$",
    re.I)
markup_pattern = re.compile(
    r"<[^<>\n]*>[>\]]*|\[?cid\s*:[^\s\]]*\]?|(?:https?|imap|file)://\S+|\[image[^\]]*\]|image removed by sender\.?"
    r"|microsoft outlook web access\s*:?|\bsignature_\d+\b|\b[\w.-]+\.(?:png|jpe?g|gif)\b"
    r"|^\s*(?:[\w&'’.,-]+\s+){0,7}logo\b(?:\s*description automatically generated[^\n]*)?\s*$"
    r"|^\s*[\w.&-]*\s*e-?mail signature\s*$", re.I)
# Attachment lists and meeting-invitation lines (zone other)
attach_pattern = re.compile(
    r"\t\s*[\d.,]+\s*(?:bytes|[KMG]B)\s*$|^\s*attachments\s*:\s*$"
    r"|^\s*<[^<>\n]+\.(?:pdf|docx?|xlsx?|pptx?|csv|zip|txt)>\s*$"
    r"|^\s*(?:microsoft teams meeting|join on your computer[^\n]*|click here to join the meeting|or call in \(audio only\)"
    r"|phone conference id\s*:[^\n]*|\+?[\d ().-]{7,},,\d+#[^\n]*|meeting id\s*:[^\n]*|passcode\s*:[^\n]*|find a local number[^\n]*"
    r"|learn more \| meeting options|join with a video conferencing device|download teams \| join on the web)\s*$", re.I)
template_min_chars, template_min_emails = 30, 20
disclaimer_pattern = re.compile(
    r"confidential|privileged|intended recipient|destinataire|unauthori[sz]ed|prohibited", re.I)

def reused_sentence(line):
    return (len(line.split()) >= 6 and line.rstrip()[-1:] in '.?!'
            and not disclaimer_pattern.search(line))

def first_name(address):
    local = str(address).split('@')[0].lower()
    parts = re.split(r'[._-]', local)
    name = re.sub(r'\d', '', parts[0])
    return name.capitalize() if len(parts) > 1 and len(name) > 2 else None

def line_spans(text):
    pos = 0
    for line in text.split('\n'):
        yield pos, pos + len(line), line
        pos += len(line) + 1

def quoted_start(body):
    starts = [m.start() for p in quote_patterns for m in re.finditer(p, body, re.I | re.M)]
    return min(starts) if starts else None

def signature_start(text, sender):
    end = None
    for s, e, line in line_spans(text):
        if signoff_pattern.match(line):
            end = e
    if end is not None:
        return end
    name = first_name(sender)
    if name:
        hits = [m.start(2) for m in re.finditer(rf"(^|[.!?,]\s+)({name})\b", text, re.M)
                if m.start(2) > len(text) / 2]
        if hits:
            return hits[-1]
    return None

def unquoted(body):
    q = quoted_start(body)
    return body if q is None else body[:q]

line_counts = Counter()
for body in pool10['body'].astype(str):
    line_counts.update({l.strip() for l in unquoted(body).split('\n') if len(l.strip()) >= template_min_chars})
template_lines = {l for l, n in line_counts.items()
                  if n >= template_min_emails and not reused_sentence(l)}

region_order = ['quoted', 'banner', 'signature', 'template']

def regions(body, sender):
    q = quoted_start(body)
    own = body if q is None else body[:q]
    out = []
    if q is not None:
        out.append((q, len(body), 'quoted'))
    for s, e, line in line_spans(own):
        for p in banner_patterns:
            m = re.search(p, line, re.I)
            if m:
                close = line.find(']', m.start()) if line[m.start()] == '[' else -1
                end = s + close + 1 if close >= 0 else min(e, s + m.start() + 300)
                out.append((s + m.start(), end, 'banner'))
                break
        if line.strip() in template_lines:
            out.append((s, e, 'template'))
    sig = signature_start(own, sender)
    if sig is not None:
        out.append((sig, len(own), 'signature'))
    out.sort(key=lambda r: region_order.index(r[2]))
    return out

# ---- Pattern zone for every Chipmunk line ----
chip_map = {'paragraph': 'authored', 'salutation': 'authored', 'closing': 'authored',
            'personal_signature': 'signature', 'mua_signature': 'banner',
            'quotation': 'quoted', 'quotation_marker': 'quoted', 'inline_headers': 'quoted'}

senders = dict(zip(pool10['msgid2'], pool10['from']))
bodies10 = dict(zip(pool10['msgid2'], pool10['body'].astype(str)))
regs = {m: regions(b, senders[m]) for m, b in bodies10.items()}

def pattern_zone(m, s, e, text):
    if markup_pattern.sub('', text).strip() == '':
        return 'markup'
    return next((lab for rs, re_, lab in regs[m] if s < re_ and rs < e), 'authored')

cmp = zones[zones['msgid2'].isin(bodies10) & (zones['text'].str.strip() != '')].copy()
cmp['patterns'] = [pattern_zone(m, s, e, t) for m, s, e, t in zip(cmp['msgid2'], cmp['start'], cmp['end'], cmp['text'])]
cmp['chipmunk_mapped'] = cmp['chipmunk'].map(chip_map).fillna('other')
cmp['agree'] = cmp['patterns'] == cmp['chipmunk_mapped']

# ---- Comparison report ----
print(f"Emails compared: {cmp['msgid2'].nunique():,} | non-blank lines: {len(cmp):,}")
print(f"Template lines (≥{template_min_chars} characters, in ≥{template_min_emails} emails): {len(template_lines):,}")
print(f"Lines where both methods give the same zone: {cmp['agree'].mean():.3f}")

print("\nAgreement per pattern zone (share of its lines Chipmunk places in the same zone):")
show(cmp.groupby('patterns')['agree'].agg(lines='size', agreement='mean').round(3))

print("\nPattern zone (rows) by Chipmunk zone (columns), line counts:")
show(pd.crosstab(cmp['patterns'], cmp['chipmunk_mapped'], margins=True))

print("\nChipmunk's own zones within each pattern zone:")
show(pd.crosstab(cmp['chipmunk'], cmp['patterns']))

print("\nSampled disagreements (up to 10 random lines per combination):")
dis = cmp[~cmp['agree']]
for (p, c), g in dis.groupby(['patterns', 'chipmunk_mapped']):
    sub = g.sample(min(len(g), 10), random_state=42)
    print(f"\npatterns: {p} | chipmunk: {c} ({len(g):,} lines)")
    show(pd.DataFrame({'msgid2': sub['msgid2'], 'chipmunk': sub['chipmunk'],
                       'text': sub['text'].str.strip().str.slice(0, 90)}))

# ---- Refined zones ----
final = zones[zones['msgid2'].isin(bodies10)].copy()
final['patterns'] = [pattern_zone(m, s, e, t) if t.strip() else 'empty'
                     for m, s, e, t in zip(final['msgid2'], final['start'], final['end'], final['text'])]
final['chipmunk_mapped'] = final['chipmunk'].map(chip_map).fillna('other')
final['zone'] = final['chipmunk_mapped']
final['rule'] = ''
blank = final['text'].str.strip() == ''

# Rule 1: closing lines are authored
is_closing = final['text'].str.match(signoff_pattern) & ~blank
r1 = is_closing & final['zone'].isin(['signature', 'banner', 'other'])
final.loc[r1, ['zone', 'rule']] = ['authored', 'closing line']

# Rule 2: authored, other or signature lines with wording found only in disclaimers are banner
disclaimer_only = re.compile(
    r"intended recipient|intended (?:solely |only )?for the (?:use|person|individual|addressee)|privileged"
    r"|destinataire|received this (?:e-?mail|message|communication) in error", re.I)
r2 = final['zone'].isin(['authored', 'other', 'signature']) & final['text'].str.contains(disclaimer_only) & ~blank
final.loc[r2, ['zone', 'rule']] = ['banner', 'disclaimer wording']

# Nearest non-blank neighbours within the same email, read after rules 1 and 2
nb_lines = final[~blank]
prev_zone = nb_lines.groupby('msgid2')['zone'].shift(1).reindex(final.index)
next_zone = nb_lines.groupby('msgid2')['zone'].shift(-1).reindex(final.index)

# Rule 3: authored lines between signature lines are signature (closing lines excepted)
ends_sentence = final['text'].str.rstrip().str[-1:].isin(['.', '?', '!'])
r3 = ((final['zone'] == 'authored') & (prev_zone == 'signature') & (next_zone == 'signature')
    & ~is_closing & ~ends_sentence & ~blank)

# Rule 4: banner or other lines that read as prose, between authored lines, are authored
prose = (final['text'].str.count(r"[A-Za-zÀ-ÿ]{2,}") >= 4) & ~final['text'].str.contains('\t')
r4 = (final['zone'].isin(['banner', 'other']) & (prev_zone == 'authored') & (next_zone == 'authored')
      & prose & (final['rule'] == '') & ~blank)

final.loc[r3, ['zone', 'rule']] = ['signature', 'inside signature block']
final.loc[r4, ['zone', 'rule']] = ['authored', 'prose inside authored text']

# Rule 5: short lines without sentence punctuation at the end of the authored text are signature.
# A run of such lines qualifies when a signature line follows it, or when it holds a name, title,
# organization or contact detail. A one-word sign-off name stays authored, and so does a two-word name
# under a closing line (or labelled closing by Chipmunk) with no credentials and nothing of the signature after it.
# Lines ending in a colon or comma never count.
sig_like = re.compile(
    r"@|\+?\(?\d{3}\)?[ .-]?\d{3}[ .-]?\d{4}\b|\b(?:ph\.?d|m\.?d|m\.?sc|mph|m\.?eng|b\.?sc|ll\.?[bm]|frcpc|faaa\w*|faap"
    r"|m\.?a|j\.?d|she/her|he/him|they/them|elle|professor|director|manager|coordinator|assistant|associate|scientist"
    r"|officer|head|chair|lead|analyst|university|institute|hospital|task force|centre|center|secretariat)\b", re.I)
words = final['text'].str.split().str.len()
# A full stop ends a sentence unless the line is short and carries credentials ("Oscar Palma, M.A., J.D.")
sentence = (final['text'].str.rstrip().str[-1:].isin(['?', '!'])
            | (ends_sentence & ((words >= 5) | ~final['text'].str.contains(sig_like))))
cand = ((final['zone'] == 'authored') & ~blank & ~is_closing & ~sentence & (words <= 12)
        & (final['chipmunk'] != 'salutation') & ~final['text'].str.rstrip().str[-1:].isin([':', ','])
        & ~final['text'].str.match(r"\s*([-•*–]|\d+[.)])\s"))
r5 = pd.Series(False, index=final.index)
for m, g in final[~blank].groupby('msgid2', sort=False):
    idx, zs, cs = list(g.index), g['zone'].tolist(), cand[g.index].tolist()
    i = 0
    while i < len(idx):
        if not cs[i]:
            i += 1
            continue
        j = i
        while j < len(idx) and cs[j]:
            j += 1
        run, nxt = idx[i:j], (zs[j] if j < len(idx) else None)
        if nxt != 'authored':
            texts = final.loc[run, 'text']
            if nxt == 'signature' or texts.str.contains(sig_like).any():
                first = texts.iloc[0]
                signoff = (i > 0 and is_closing[idx[i - 1]]) or final.at[run[0], 'chipmunk'] == 'closing'
                alone = len(run) == 1 and nxt != 'signature'
                if len(first.split()) <= 1 or (signoff and alone and len(first.split()) <= 2 and not sig_like.search(first)):
                    run = run[1:]
                r5[run] = True
        i = j
final.loc[r5, ['zone', 'rule']] = ['signature', 'signature lines after the message']

# Pattern zones take precedence, in this order: markup, attachment and meeting lines (other), banner, quoted
attach = final['text'].str.contains(attach_pattern) & ~blank
for z, hit, name in [('markup', final['patterns'] == 'markup', 'pattern markup'),
                     ('other', attach, 'attachment or meeting line'),
                     ('banner', final['patterns'] == 'banner', 'pattern banner'),
                     ('quoted', final['patterns'] == 'quoted', 'pattern quoted')]:
    final.loc[hit & (final['zone'] != z), 'rule'] = name
    final.loc[hit, 'zone'] = z
final.loc[blank, ['zone', 'rule']] = ['empty', '']
final.to_parquet(zones_path, index=False)

# ---- Refined-zone report ----
nb = final[final['zone'] != 'empty']
print("\nRefined zones (non-blank lines):")
show(nb['zone'].value_counts().rename('lines').to_frame())

print("\nLines changed from Chipmunk's zone, by rule:")
show(nb[nb['rule'] != ''].groupby('rule').agg(lines=('zone', 'size'),
     changes=('zone', lambda z: '; '.join(f"{k} {v}" for k, v in
              (nb.loc[z.index, 'chipmunk_mapped'] + ' → ' + z).value_counts().items()))))

print("\nSampled lines per rule (up to 10):")
for r, g in nb[nb['rule'].isin(['closing line', 'disclaimer wording', 'inside signature block',
                                'prose inside authored text', 'signature lines after the message',
                                'attachment or meeting line'])].groupby('rule'):
    sub = g.sample(min(len(g), 10), random_state=42)
    print(f"\n{r} ({len(g):,} lines)")
    show(pd.DataFrame({'msgid2': sub['msgid2'], 'chipmunk': sub['chipmunk'], 'zone': sub['zone'],
                       'text': sub['text'].str.strip().str.slice(0, 90)}))

print(f"\nSaved with columns {list(final.columns)} to: {zones_path}")